In [ ]:
from torch.nn import *
from math import log2
import torch

torch.set_default_tensor_type(torch.cuda.FloatTensor)


def unpack(a, n, m=2 ** torch.arange(16)):
    return (a[..., None] & m[:n]).bool().float()


def center(a):
    return a.mul(2.0).add_(-1.0)


def dat(k, c, s=1, n=1, l=1, rx=1.0, rk=1.0, rb=1.0):
    l = 2 ** int(log2(c * s * (k + 1) * l))
    y = torch.randint(c * s, (n, l))
    return (
        torch.randn(n, c * s, k)
        .mul_(rx * rb * l ** (1 / k))[torch.arange(n)[:, None], y]
        .view(n * l, 1, k)
        .baddbmm(torch.randn(n * l, 1, k).mul_(rx), torch.randn(n, c * s, k, k)[torch.arange(n)[:, None], y].view(n * l, k, k))
        .view(n, l, k),
        unpack(y // s, log2(c).__ceil__()),
    )


class Attention(Sequential):
    def forward(s, q, k=None, v=None):
        if k is None:
            k = q
        return (s[0](q) @ s[1](k).transpose(-2, -1)).softmax(-1) @ s[2](k if v is None else v)


class Scale(Module):
    def __init__(s, w=..., b=None):
        super().__init__()
        s.w = Parameter(torch.ones(())) if w is ... else w
        s.b = Parameter(torch.zeros(())) if b is ... else b

    def forward(s, x):
        if s.w is not None:
            x = x * s.w
        if s.b is not None:
            x += s.b
        return x


class Norm(Module):
    @staticmethod
    def forward(x):
        return x * x.norm(dim=-1, keepdim=True)

In [ ]:
k1 = 6
k2 = 2
k = k1 + k2
kh = k + 1
h = 16
c1 = 2 ** k1
c2 = 2 ** k2
c = c1 * c2
ch = 2 ** kh
I = center(unpack(torch.arange(ch), kh))
M = Attention(Scale(), Sequential(Linear(k + k1, h), Hardswish(1), Linear(h, kh), Norm()), Sequential(Linear(k + k1, h), Hardswish(1)))
m = Attention(
    Sequential(Linear(k, h), Hardswish(1), Linear(h, h), Norm()), Sequential(Linear(h, h), Hardswish(1), Linear(h, h), Norm(), Scale()), Sequential(Linear(h, h), Hardswish(1), Linear(h, k1))
)
o = torch.optim.Adam((*M.parameters(), *m.parameters()))
O = torch.optim.lr_scheduler.OneCycleLR(o, 0.0005, 100000, pct_start=0.01, div_factor=9, final_div_factor=99)
print(M, m, sum(i.numel() for i in o.param_groups[0]["params"]))
for t in range(O.total_steps // 1000):
    l = []
    a = []
    for t in range(10):
        for t in range(100):
            x, y = dat(k, c1, c2, n=2)
            t = torch.cat((x, y), 2)
            t = M(I, t, t)
            x = m(x, t, t)
            t = functional.binary_cross_entropy_with_logits(x, y)
            t.backward()
            o.step()
            O.step()
            o.zero_grad()
        l.append(t)
        a.append((x.signbit() != y).all(-1).sum() / y.shape[:-1].numel())
    print(sum(l) / len(l), sum(a) / len(a), o.param_groups[0]["lr"])